# Introducción al RAG Agéntico aplicado a la Ingeniería de Software

Taller de **2 horas**, 100% sin programar.
Aprenderás la técnica **RAG** (y su versión **agéntica**) para construirlo, paso a paso.

Cada vez que alguien llega a un proyecto —o retoma código viejo— pierde horas
entendiendo cómo funciona. En este taller construiremos un asistente que resuelve eso:
lee el **código**, la **documentación**, el **UML** y las **pruebas** de un sistema de
ventas, y responde citando la fuente exacta. Es una herramienta de **mejora de procesos (SPI)**.

### Qué aprenderás
- Qué es RAG y por qué la IA por sí sola no conoce tu código.
- A indexar un proyecto (código, documentación, UML y pruebas).
- A construir un **agente** que decide dónde buscar para responder.

**Cómo usar este notebook:** ejecuta cada celda en orden con el botón ▶ (o `Shift + Enter`).
No necesitas leer ni editar el código: solo ejecuta celdas, pega tu llave y escribe preguntas.


## Parte 1 · Bienvenida, entorno y qué es RAG  (25 min)

### El problema (de ingeniería de software)
Entender un proyecto ajeno cuesta tiempo: leer miles de líneas, preguntar a quien
está disponible, descifrar código sin documentar. Esto afecta el *onboarding* de
personas nuevas y el mantenimiento de *código legado*.

Una IA que "escribe texto" (un *modelo de lenguaje*) no conoce tu proyecto:
si le preguntas por tu código, puede inventar la respuesta. Herramientas como
**GitHub Copilot Chat**, **Sourcegraph** o **Cursor** resuelven esto con una técnica: RAG.

### La idea de RAG
**RAG** (*Retrieval-Augmented Generation*) funciona en dos pasos:
1. **Recuperar:** buscar los fragmentos de tu proyecto más relevantes a la pregunta.
2. **Generar:** dar esos fragmentos al modelo para que responda con base en ellos (y cite la fuente).

Primero preparamos el entorno (instalar, pegar la llave) y luego lo aplicamos.


In [ ]:
# @title 1.1 Instalar dependencias (ejecuta una sola vez)
# Instala LlamaIndex y los componentes de Gemini. Tarda 1-2 minutos.

!pip install -q llama-index==0.14.25 llama-index-llms-gemini==0.6.2 llama-index-embeddings-gemini==0.4.2 google-generativeai==0.8.6 ipywidgets==8.1.9 llama-index-readers-file 2>/dev/null
print("✅ Dependencias instaladas.")


In [ ]:
# @title 1.2 Pega tu llave gratuita de Gemini
# 1) Ve a https://aistudio.google.com/app/apikey
# 2) Inicia sesión con tu cuenta de Google y crea una API key.
# 3) Ejecuta esta celda y pega la llave cuando te la pida.

import os
import getpass

os.environ["GOOGLE_API_KEY"] = getpass.getpass("Pega aquí tu API key de Gemini y presiona Enter: ")

print("✅ Llave configurada.")


In [ ]:
# @title 1.3 Configurar el modelo (el que "escribe" y el que "busca")
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

from llama_index.core import Settings
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.gemini import GeminiEmbedding

# Modelo que genera las respuestas (flash-lite: gratuito y con cuota holgada)
Settings.llm = Gemini(model="models/gemini-flash-lite-latest")

# Modelo que convierte texto en números para poder buscar por significado
Settings.embed_model = GeminiEmbedding(model_name="models/gemini-embedding-001")

print("✅ Listo. Modelo configurado con Gemini.")


## Parte 2 · Indexar el código  (30 min)

Para que la IA pueda responder sobre el proyecto, primero hay que **capturar su
conocimiento**: leer el código, la documentación y el UML, dividirlos en bloques
pequeños (*chunks*) y convertirlos en números (*embeddings*) que permitan buscar
por significado.

Es lo mismo que hace Sourcegraph o Copilot por dentro: construyen una base de
conocimiento consultable del proyecto. Ejecuta las siguientes dos celdas.


In [ ]:
# @title 2.1 Cargar el código de ejemplo
import os

REPO_URL = "https://github.com/cossio-udg/taller-rag-agentico.git"
RUTA = "taller-rag-agentico/data/repo_ejemplo"

if not os.path.isdir(RUTA):
    print("Descargando el repositorio de ejemplo...")
    !git clone {REPO_URL}
else:
    print("El código de ejemplo ya está disponible.")

print("\n📁 Archivos que se van a indexar:")
for raiz, dirs, archivos in os.walk(RUTA):
    for a in sorted(archivos):
        print("   -", os.path.join(raiz, a))


In [ ]:
# @title 2.2 Indexar (leer, dividir en bloques y guardar)
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter

# 1) Leer todos los archivos .py, .md y .puml del proyecto
documentos = SimpleDirectoryReader(
    RUTA, recursive=True, required_exts=[".py", ".md", ".puml"], filename_as_id=True
).load_data()
print(f"📄 Documentos leídos: {len(documentos)}")

# 2) Dividir en bloques (chunks) para buscar por partes
divisor = SentenceSplitter(chunk_size=400, chunk_overlap=50)
nodos = divisor.get_nodes_from_documents(documentos)
print(f"🧩 Bloques creados: {len(nodos)}")

# 3) Crear el índice (convierte los bloques en números y los guarda)
indice = VectorStoreIndex(nodos)
print("✅ Índice listo. Ya puedes hacer preguntas.")


## Parte 3 · Hacer preguntas y ver citas  (30 min)

Ahora haz preguntas como lo haría un ingeniero que llega nuevo al proyecto.
Verás la respuesta **junto a la fuente** (archivo y relevancia): eso es trazabilidad.

**Preguntas clave (sistema de ventas):**
- ¿Cómo se valida el stock al crear un pedido?
- ¿Qué regla de descuento aplica el sistema?
- ¿Qué relación hay entre Cliente y Pedido?
- Traza el requisito R8 desde su definición hasta su prueba.


In [ ]:
# @title 3.1 Preguntar al sistema (sin escribir código)
import ipywidgets as widgets
from IPython.display import display, clear_output

def condensar_pregunta(historial, pregunta):
    """Reescribe la pregunta usando el contexto de la conversación previa."""
    if not historial:
        return pregunta
    contexto = "\n".join(f"Pregunta: {u}\nRespuesta: {a}" for u, a in historial)
    prompt = (
        "Tienes una conversación previa. Reescribe la NUEVA pregunta para que sea "
        "completa por sí sola, usando el contexto. Responde SOLO con la pregunta reescrita.\n\n"
        f"Conversación previa:\n{contexto}\n\n"
        f"Nueva pregunta: {pregunta}\n\n"
        "Pregunta reescrita:"
    )
    return Settings.llm.complete(prompt).text.strip()

def crear_consola(motor, titulo="Pregunta al sistema", fuentes=None, chat=False):
    historial = []
    caja = widgets.Textarea(
        value="",
        placeholder="Escribe aquí tu pregunta en lenguaje natural...",
        layout=widgets.Layout(width="100%", height="70px"),
    )
    boton = widgets.Button(description="Preguntar", button_style="primary")
    salida = widgets.Output()

    def al_pulsar(_):
        with salida:
            clear_output()
            pregunta = caja.value.strip()
            if not pregunta:
                print("⚠️ Escribe una pregunta primero.")
                return
            pregunta_final = condensar_pregunta(historial, pregunta) if chat else pregunta
            print("🔍 Buscando en el proyecto...")
            respuesta = motor.query(pregunta_final)
            seleccion = respuesta.metadata.get("selector_result") if respuesta.metadata else None
            if seleccion is not None:
                elegido = seleccion.selections[0]
                nombre = fuentes[elegido.index] if (fuentes and 0 <= elegido.index < len(fuentes)) else f"fuente {elegido.index}"
                print(f"\n🧠 El agente decidió buscar en: {nombre}")
                if elegido.reason:
                    print(f"   Motivo: {elegido.reason}")
            print("\n📝 Respuesta:\n")
            print(respuesta)
            nodos = getattr(respuesta, "source_nodes", [])
            if nodos:
                print("\n📎 Fuentes (trazabilidad):")
                for n in nodos:
                    archivo = n.node.metadata.get("file_name", "?")
                    print(f"   • {archivo}  (relevancia: {n.score:.2f})")
            if chat:
                historial.append((pregunta, str(respuesta)))
                caja.value = ""

    boton.on_click(al_pulsar)
    display(widgets.VBox([widgets.HTML(f"<h3>{titulo}</h3>"), caja, boton, salida]))

motor = indice.as_query_engine(similarity_top_k=3, response_mode="compact")
crear_consola(motor, "Pregunta al RAG clásico")


## Parte 4 · El agente decide dónde buscar  (25 min)

Hasta ahora tenemos un RAG **clásico**: siempre busca en un mismo montón.

Un RAG **agéntico** emula la decisión de un ingeniero: según la tarea, elige si
buscar en el **código**, la **documentación**, el **UML** o las **pruebas**.
Recuerda: cada fuente del agente **es** un RAG; lo "agéntico" es solo la capa que decide cuál usar.
Además, este agente tiene **memoria**: puedes hacer preguntas de seguimiento
(por ejemplo, «¿y cuál viene después de enviado?»).

Ejecuta la siguiente celda y haz preguntas de distintos tipos para ver cómo
el agente decide por sí solo a qué fuente acudir.

**Preguntas por fuente:**
- Código: `¿Qué hace la función confirmar en ServicioVentas?`
- Documentación: `¿En qué capas está dividido el proyecto y por qué?`
- UML: `¿Cuáles son los estados de un pedido?`
- Pruebas: `¿Qué prueba que cancelar un pedido confirmado repone stock?`


In [ ]:
# @title 4.1 Crear el agente que decide dónde buscar
import os
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.tools import QueryEngineTool

# Fuente A: código fuente (carpetas dominio, servicios, persistencia, api)
CARPETAS_CODIGO = ["dominio", "servicios", "persistencia", "api"]
docs_codigo = []
for carpeta in CARPETAS_CODIGO:
    ruta = os.path.join(RUTA, carpeta)
    docs_codigo += SimpleDirectoryReader(
        ruta, recursive=True, required_exts=[".py"], filename_as_id=True
    ).load_data()
indice_codigo = VectorStoreIndex(divisor.get_nodes_from_documents(docs_codigo))

# Fuente B: documentación y requisitos (archivos .md en la raíz)
docs_docs = SimpleDirectoryReader(
    RUTA, recursive=False, required_exts=[".md"], filename_as_id=True
).load_data()
indice_docs = VectorStoreIndex(divisor.get_nodes_from_documents(docs_docs))

# Fuente C: UML (archivos .puml en modelo/)
ruta_modelo = os.path.join(RUTA, "modelo")
docs_uml = SimpleDirectoryReader(
    ruta_modelo, recursive=True, required_exts=[".puml"], filename_as_id=True
).load_data()
indice_uml = VectorStoreIndex(divisor.get_nodes_from_documents(docs_uml))

# Fuente D: pruebas (archivos dentro de tests/)
ruta_tests = os.path.join(RUTA, "tests")
docs_pruebas = SimpleDirectoryReader(
    ruta_tests, recursive=True, required_exts=[".py"], filename_as_id=True
).load_data()
indice_pruebas = VectorStoreIndex(divisor.get_nodes_from_documents(docs_pruebas))

motor_codigo = indice_codigo.as_query_engine(similarity_top_k=3, response_mode="compact")
motor_docs = indice_docs.as_query_engine(similarity_top_k=3, response_mode="compact")
motor_uml = indice_uml.as_query_engine(similarity_top_k=3, response_mode="compact")
motor_pruebas = indice_pruebas.as_query_engine(similarity_top_k=3, response_mode="compact")

herramienta_codigo = QueryEngineTool.from_defaults(
    query_engine=motor_codigo,
    description="Responde sobre la implementación: funciones, clases y lógica del código fuente.",
)
herramienta_docs = QueryEngineTool.from_defaults(
    query_engine=motor_docs,
    description="Responde sobre documentación, requisitos y decisiones de diseño.",
)
herramienta_uml = QueryEngineTool.from_defaults(
    query_engine=motor_uml,
    description="Responde sobre los diagramas UML: clases, relaciones y máquina de estados.",
)
herramienta_pruebas = QueryEngineTool.from_defaults(
    query_engine=motor_pruebas,
    description="Responde sobre las pruebas automatizadas y el comportamiento esperado.",
)

# El selector (un LLM) decide a qué fuente enviar cada pregunta, como un ingeniero
agente = RouterQueryEngine(
    selector=LLMSingleSelector.from_defaults(),
    query_engine_tools=[herramienta_codigo, herramienta_docs, herramienta_uml, herramienta_pruebas],
)

print("✅ Agente listo. Decide solo entre código, documentación, UML o pruebas.")


In [ ]:
# @title 4.2 Preguntar al agente (él decide dónde buscar)
crear_consola(agente, "Pregunta al RAG agéntico", fuentes=["código", "documentación", "UML", "pruebas"], chat=True)


## Parte 5 · Cierre y siguiente paso  (10 min)

¡Listo! Tienes un agente de RAG funcionando sobre un proyecto real.

### Qué lograste (y por qué es mejora de procesos)
- Centralizaste el conocimiento del proyecto en una base consultable.
- Redujiste el tiempo de *onboarding*: preguntas en lenguaje natural, respuestas con fuente.
- Ganaste *trazabilidad* entre requisitos, código, UML, documentación y pruebas.

### Para reutilizarlo
Guarda tu copia en Drive: **Archivo → Guardar una copia en Drive**.
La próxima vez, abre tu copia y ejecuta las celdas en orden.


In [ ]:
# @title 5.1 Ajustar cuántos bloques recupera
# Cambia este número (prueba 1, 3 o 5) y vuelve a ejecutar la celda.
top_k = 5

motor_ajustado = indice.as_query_engine(similarity_top_k=top_k, response_mode="compact")
crear_consola(motor_ajustado, "Agente ajustado")


### 🎉 Resumen

1. Entendiste el problema de SE: entender un proyecto ajeno cuesta tiempo.
2. Capturaste el conocimiento del proyecto (código + documentación + UML + requisitos).
3. Hiciste preguntas y viste la respuesta con su fuente (trazabilidad).
4. Convertiste el RAG en un **agente** que elige entre código, docs, UML y pruebas.
5. Ajustaste el comportamiento y dejaste tu agente listo para reutilizar.

**Siguiente paso sugerido:** apunta tu agente a tu propio repositorio cambiando
la variable `RUTA` en la celda 2.1. Esa es la práctica de mejora de procesos que
puedes llevar a tu equipo.
